# Pipeline Dự Báo Giao Thông (utils + ml + rl)


Notebook này tổng hợp pipeline dự báo: nhận `request_time` từ app, lấy 12 timestep gần nhất, dự báo timestep tiếp theo (+15 phút), và chỉ giữ kết quả trong khung 09:15-21:15.

## Flow Chart Pipeline


```mermaid
flowchart TD
    A[App gửi request_time + corridor_id] --> B[Tải dữ liệu lịch sử bằng load_bulk_corridor_data]
    B --> C{Đủ 12 timestep liên tục?}
    C -- Không --> C1[Bỏ segment]
    C -- Có --> D[Tiền xử lý: encode + scale theo artifacts]
    D --> E[RL model suy luận Q-values]
    E --> F[Chọn lớp có Q-value lớn nhất]
    F --> G[Tính Forecast_For_Time = Window_End_Time + 15 phút]
    G --> H{Forecast_For_Time trong 09:15-21:15?}
    H -- Không --> H1[Loại khỏi kết quả]
    H -- Có --> I[Ghi output theo schema chuẩn]
    I --> J[Trả DataFrame/CSV/API response]
```


### Ghi chú nghiệp vụ


- `Window_End_Time` là mốc dữ liệu cuối dùng làm đầu vào.
- `Forecast_For_Time` mới là mốc thời gian được dự báo.
- Nếu App muốn dự báo mốc `t`, dữ liệu cần đủ để tạo cửa sổ kết thúc ở `t - 15 phút`.

## Input/Output Schema


### Input (từ App / API)


| Trường | Kiểu dữ liệu | Bắt buộc | Mô tả | Ví dụ |
|---|---|---|---|---|
| `corridor_id` | `int64` | Có | Mã corridor cần dự báo | `646713380690000556` |
| `request_time` | `datetime` | Có | Thời điểm App yêu cầu dự báo | `2026-04-08 20:00:00` |


### Xử lý nội bộ


- Truy vấn lịch sử dữ liệu đến `request_time`.
- Chọn 12 timestep liên tục gần nhất (chu kỳ 15 phút).
- Dự báo timestep kế tiếp bằng RL model.
- Lọc kết quả theo khung `09:15 - 21:15`.


### Output (trả về backend/app)


| Trường | Kiểu dữ liệu | Mô tả |
|---|---|---|
| `Segment_ID` | `int64` | Định danh đoạn đường |
| `Request_Time` | `datetime` | Thời điểm App gửi request |
| `Window_End_Time` | `datetime` | Mốc cuối của 12 timestep đầu vào |
| `Forecast_For_Time` | `datetime` | Mốc thời gian được dự báo (`Window_End_Time + 15m`) |
| `Dự báo (15p tới)` | `string` | Mức độ giao thông dự báo |
| `Q-Values (Kỳ vọng)` | `string/array` | Điểm kỳ vọng cho 6 mức lớp |

## Mục Tiêu Chức Năng


Pipeline này phục vụ nghiệp vụ dự báo giao thông theo yêu cầu thời gian thực từ App, với nguyên tắc:


- Khi nhận `request_time`, hệ thống tự truy xuất 12 timestep gần nhất cho từng segment.
- Dự báo mức ùn tắc cho timestep kế tiếp (`+15 phút`).
- Chỉ trả kết quả khi `Forecast_For_Time` nằm trong khung vận hành `09:15 - 21:15`.


### Kết quả mong muốn


- Dự báo ổn định theo từng segment trong corridor.
- Dễ tích hợp vào API backend với cấu trúc output rõ ràng.
- Tránh dự báo sai ngữ cảnh ngoài khung nghiệp vụ.

## 1) Thiết lập và import


- `utils.data_loader`: tải dữ liệu corridor


- `ml.traffic_model`: kiến trúc mô hình


- `rl.inference_rl`: predictor và pipeline theo request

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd

# Xac dinh root du an dua tren cau truc thu muc thay vi ten folder
cwd = Path.cwd().resolve()
search_roots = [cwd] + list(cwd.parents)
PROJECT_ROOT = next(
    (p for p in search_roots if (p / 'src').exists() and (p / 'artifacts').exists()),
    cwd,
 )

if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

ROOT = PROJECT_ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Thu muc lam viec:', Path.cwd().resolve())
print('PROJECT_ROOT:', ROOT)

Thu muc lam viec: C:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\DATN\project_folder\traffic-ioc\ai-core
PROJECT_ROOT: C:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\DATN\project_folder\traffic-ioc\ai-core


In [2]:
from src.data_access import get_segments_in_corridor
from src.ml.feature_contract import WINDOW_SIZE_DEFAULT, WINDOW_STEP_MINUTES
from src.rl.inference.predictor import (
    RLTrafficPredictor,
    forecast_for_request,
    is_within_forecast_window,
)

print('Đã import thành công')

ModuleNotFoundError: No module named 'torch'

## 2) Cấu hình nghiệp vụ


Bạn có thể đổi `REQUEST_TIME` mỗi khi App gửi yêu cầu.


Quy tắc:


- Pipeline tự lấy 12 timestep gần nhất (`<= request_time`)


- Dự báo cho mốc tiếp theo (+15 phút)


- Chỉ nhận dự báo nếu `forecast_for_time` nằm trong 09:15-21:15

In [ ]:
# Cau hinh nghiep vu
CORRIDOR_ID = 646713380690000556
REQUEST_TIME = '2026-04-08 20:00:00'  # Thoi diem App gui yeu cau

# Tu dong tim artifact/model RL trong workspace
model_candidates = [
    ROOT / 'artifacts' / 'rl' / 'checkpoints' / 'best_rl_agent.pt',
    ROOT / 'best_rl_agent.pt',
]
artifact_candidates = [
    ROOT / 'artifacts' / 'rl' / 'preprocessing' / 'rl_pure_preprocessing_artifacts.pkl',
    ROOT / 'preprocessing_artifacts.pkl',
]

MODEL_PATH = next((str(p) for p in model_candidates if p.exists()), None)
ARTIFACTS_PATH = next((str(p) for p in artifact_candidates if p.exists()), None)

print('ROOT dang dung:', ROOT)
print('Model candidates:', [str(p) for p in model_candidates])
print('Artifact candidates:', [str(p) for p in artifact_candidates])

if MODEL_PATH is None:
    raise FileNotFoundError(
        'Khong tim thay model RL. Kiem tra thu muc artifacts/rl/checkpoints hoac cap nhat MODEL_PATH.'
    )
if ARTIFACTS_PATH is None:
    raise FileNotFoundError(
        'Khong tim thay preprocessing artifacts RL. Kiem tra thu muc artifacts/rl/preprocessing hoac cap nhat ARTIFACTS_PATH.'
    )

SEGMENT_IDS = get_segments_in_corridor(CORRIDOR_ID)
if not SEGMENT_IDS:
    raise ValueError(f'Khong tim thay segment nao cho corridor_id={CORRIDOR_ID}')

print('Corridor:', CORRIDOR_ID)
print('So segment trong corridor:', len(SEGMENT_IDS))
print('Thoi diem request:', REQUEST_TIME)
print('Model path:', MODEL_PATH)
print('Artifacts path:', ARTIFACTS_PATH)
print('Nam trong khung du bao:', is_within_forecast_window(pd.to_datetime(REQUEST_TIME)))

ROOT dang dung: /app
Model candidates: ['/app/artifacts/rl/checkpoints/best_rl_agent.pt', '/app/best_rl_agent.pt']
Artifact candidates: ['/app/artifacts/rl/preprocessing/rl_pure_preprocessing_artifacts.pkl', '/app/preprocessing_artifacts.pkl']
Corridor: 646713380690000556
So segment trong corridor: 241
Thoi diem request: 2026-04-08 20:00:00
Model path: /app/artifacts/rl/checkpoints/best_rl_agent.pt
Artifacts path: /app/artifacts/rl/preprocessing/rl_pure_preprocessing_artifacts.pkl
Nam trong khung du bao: True


In [ ]:
from pathlib import Path
import joblib
import torch

from src.ml.models.traffic_model import TrafficCongestionModel

ckpt_dir = ROOT / 'artifacts' / 'rl' / 'checkpoints'
prep_dir = ROOT / 'artifacts' / 'rl' / 'preprocessing'

checkpoints = sorted(ckpt_dir.glob('*.pt'))
artifacts = sorted(prep_dir.glob('*.pkl'))

print('So checkpoints:', len(checkpoints))
print('So preprocessing artifacts:', len(artifacts))

compatible_pairs = []
for pkl in artifacts:
    art = joblib.load(pkl)
    vocab_sizes = {col: len(enc.classes_) for col, enc in art['encoders'].items()}
    model = TrafficCongestionModel(vocab_sizes=vocab_sizes)
    for ckpt in checkpoints:
        try:
            state = torch.load(ckpt, map_location='cpu')
            model.load_state_dict(state)
            compatible_pairs.append((str(ckpt), str(pkl)))
            print('OK:', ckpt.name, '<->', pkl.name)
        except Exception:
            pass

if not compatible_pairs:
    print('Khong tim thay cap checkpoint/artifacts nao tuong thich.')
else:
    print('\nTong so cap tuong thich:', len(compatible_pairs))
    print('Cap dau tien:', compatible_pairs[0])

So checkpoints: 15
So preprocessing artifacts: 5
OK: best_rl_agent_warmstart_warmstart_manual_h15.pt <-> rl_pure_preprocessing_artifacts_pure_balanced_gpu.pkl
OK: best_rl_agent_warmstart_warmstart_manual_h30.pt <-> rl_pure_preprocessing_artifacts_pure_balanced_gpu.pkl
OK: best_rl_agent_warmstart_warmstart_manual_h15.pt <-> rl_pure_preprocessing_artifacts_pure_fast_gpu.pkl
OK: best_rl_agent_warmstart_warmstart_manual_h30.pt <-> rl_pure_preprocessing_artifacts_pure_fast_gpu.pkl
OK: best_rl_agent_warmstart_warmstart_manual_h15.pt <-> rl_pure_preprocessing_artifacts_pure_full.pkl
OK: best_rl_agent_warmstart_warmstart_manual_h30.pt <-> rl_pure_preprocessing_artifacts_pure_full.pkl
OK: best_rl_agent_warmstart_warmstart_manual_h15.pt <-> rl_pure_preprocessing_artifacts_pure_full_gpu.pkl
OK: best_rl_agent_warmstart_warmstart_manual_h30.pt <-> rl_pure_preprocessing_artifacts_pure_full_gpu.pkl

Tong so cap tuong thich: 8
Cap dau tien: ('/app/artifacts/rl/checkpoints/best_rl_agent_warmstart_warms

## 3) Khởi tạo predictor (RL + ML artifacts)

In [ ]:
predictor = RLTrafficPredictor(
    model_path=MODEL_PATH,
    artifacts_path=ARTIFACTS_PATH,
)

print('Predictor đã sẵn sàng')
print('Lớp mô hình:', type(predictor.agent_net).__name__)
print('Thiết bị chạy:', predictor.device)

📥 Đang nạp Preprocessing Artifacts...
🧠 Đang nạp Tác tử RL từ: /app/artifacts/rl/checkpoints/best_rl_agent.pt...


RuntimeError: Error(s) in loading state_dict for TrafficCongestionModel:
	size mismatch for embeddings.osm_highway_type.weight: copying a param with shape torch.Size([4, 8]) from checkpoint, the shape in current model is torch.Size([3, 8]).
	size mismatch for embeddings.district.weight: copying a param with shape torch.Size([12, 8]) from checkpoint, the shape in current model is torch.Size([6, 8]).
	size mismatch for embeddings.day_of_week.weight: copying a param with shape torch.Size([7, 8]) from checkpoint, the shape in current model is torch.Size([4, 8]).
	size mismatch for classifier.3.weight: copying a param with shape torch.Size([6, 64]) from checkpoint, the shape in current model is torch.Size([4, 64]).
	size mismatch for classifier.3.bias: copying a param with shape torch.Size([6]) from checkpoint, the shape in current model is torch.Size([4]).

## 4) Chạy pipeline dự báo theo request


Ô này sẽ:


1. Gọi data_loader để lấy dữ liệu lịch sử corridor


2. Tự động cắt cửa sổ 12 timestep/segment


3. Dự báo +15 phút


4. Lọc theo khung 09:15-21:15

In [ ]:
df_results = forecast_for_request(
    predictor=predictor,
    segment_ids=SEGMENT_IDS,
    request_time=REQUEST_TIME,
    lookback_steps=WINDOW_SIZE_DEFAULT,
    resample_minutes=WINDOW_STEP_MINUTES,
)

print('Tong so du bao hop le:', len(df_results))
df_results.head(10)

## 5) Tổng hợp nhanh kết quả

In [ ]:
if df_results.empty:
    print('Không có segment nào đạt điều kiện cho thời điểm request này.')
else:
    summary = (
        df_results.groupby('Dự báo (15p tới)').size()
        .sort_values(ascending=False)
        .rename('segment_count')
        .reset_index()
    )
    display(summary)

    print('Kiểm tra chân trời dự báo (phút):')
    horizon = (
        pd.to_datetime(df_results['Forecast_For_Time'])
        - pd.to_datetime(df_results['Window_End_Time'])
    ).dt.total_seconds().div(60)
    print(horizon.value_counts().sort_index())

## 6) Lưu output cho backend/app


File CSV có thể được backend đọc để trả về API.

In [ ]:
output_dir = Path('reports')
output_dir.mkdir(parents=True, exist_ok=True)

safe_request = pd.to_datetime(REQUEST_TIME).strftime('%Y%m%d_%H%M%S')
out_path = output_dir / f'rl_forecast_{CORRIDOR_ID}_{safe_request}.csv'

df_results.to_csv(out_path, index=False, encoding='utf-8-sig')
print('Đã lưu:', out_path)

## 7) Hàm wrapper để tích hợp với API App


Ô này đóng gói một hàm có thể gọi trực tiếp từ service layer.

In [ ]:
def run_request_pipeline(corridor_id: int, request_time: str) -> pd.DataFrame:
    segment_ids = get_segments_in_corridor(corridor_id)
    if not segment_ids:
        return pd.DataFrame()

    model_candidates = [
        ROOT / 'artifacts' / 'rl' / 'checkpoints' / 'best_rl_agent.pt',
        ROOT / 'best_rl_agent.pt',
    ]
    artifact_candidates = [
        ROOT / 'artifacts' / 'rl' / 'preprocessing' / 'rl_pure_preprocessing_artifacts.pkl',
        ROOT / 'preprocessing_artifacts.pkl',
    ]

    model_path = next((str(p) for p in model_candidates if p.exists()), None)
    artifacts_path = next((str(p) for p in artifact_candidates if p.exists()), None)

    if model_path is None or artifacts_path is None:
        raise FileNotFoundError(
            f'Khong tim thay model/artifacts RL voi ROOT={ROOT}. Kiem tra lai duong dan trong artifacts/rl.'
        )

    predictor = RLTrafficPredictor(
        model_path=model_path,
        artifacts_path=artifacts_path,
    )
    df = forecast_for_request(
        predictor=predictor,
        segment_ids=segment_ids,
        request_time=request_time,
        lookback_steps=WINDOW_SIZE_DEFAULT,
        resample_minutes=WINDOW_STEP_MINUTES,
    )
    return df

# Vi du:
# df_api = run_request_pipeline(646713380690000556, '2026-04-08 20:00:00')
# display(df_api.head())